# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Ranking / scoring.**

The lane (Refresh / Content Opportunity Scoring) asks "which page should an editor review first?" That is a ranking problem: assign a priority score to every page, then sort by score descending. The top K pages become the editor's queue. This is distinct from classification (which only labels yes/no) because the editor works through a ranked list, not a binary pile.

## 2. Target or proxy

**Proxy label:** a page is positive if trend_direction == "down" (is_declining_label). This label is a proxy — it is computed from the current snapshot's trend_pct, not from a future observed outcome. The starter pipeline uses it to teach the workflow, and it does identify pages worth attention. However, a stronger capstone would define a future-looking label (e.g. features from prior 90 days predicting decline or recovery over the next 30 days).

**Binary simplification:** the starter collapses 5 trend directions into 1 label (down vs not-down). This loses nuance — "new" pages behave differently from "stable" ones — but it creates a clear yes/no target to learn against.

## 3. Success metric

**Precision@50** — of the top 50 pages the system recommends, how many are genuinely declining? This metric matches the real workflow: an editor has capacity to review ~50 pages per week and needs confidence that time spent on the queue is well used. The baseline achieves 0.240 (12 of 50 correct); the model achieves 0.740 (37 of 50). The lift is more important than the absolute number — improvement over the simplest possible method is what proves the model adds value.

## 4. The unit of analysis, as a real dataframe

**One row = one content item (page).** Each row is a pseudonymized page with its trailing-90-day metrics, metadata, and trend direction. The cell below loads the starter CSV and shows the columns that matter for the refresh scoring lane.

In [ ]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

cols = ["content_id", "client_id", "impressions_90d", "avg_position",
        "ctr", "word_count", "content_age_days", "trend_direction"]
display(df[cols].head())

print(f"Rows: {len(df):,} | Clients: {df['client_id'].nunique()} | Each row = one page")

## 5. Why ML beats a fixed rule here

Because the signals interact in ways too tangled for hand-written if-statements. For example, the declining rate depends on BOTH a page's age tier AND its position tier simultaneously — and the pattern isn't monotonic. Young pages in top positions decline at 71%, while old pages in deep positions decline at only 21%. A fixed rule with fixed thresholds cannot capture these interactions without ballooning into hundreds of case-specific branches.

The pipeline proves this empirically: the hand-written baseline (a weighted sum of 4 scores) scores 0.240 Precision@50, while the random forest scores 0.740. ML captures non-linear relationships and interactions that are impractical to code by hand.

In [ ]:
# Show that declining rate varies by BOTH age_tier AND position_tier
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

ct = pd.crosstab(df["age_tier"], df["position_tier"],
                 values=df["trend_direction"] == "down", aggfunc="mean")
print("Declining rate by age tier x position tier")
print(ct.round(3).to_string())
print()
print("A fixed rule cannot capture this interaction without hundreds of branches.")
print("ML models learn these patterns automatically from data.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.